# 서울 공공도서관 ShelfAlign 도입 적합도 분석

**ShelfAlign**은 도서관 서가 오배열 도서를 탐지하고 정정 위치를 안내하는 서비스입니다.  
이 노트북은 MVP 대상 도서관 선정을 위해 서울 공공도서관을 데이터 기반으로 평가합니다.

### 평가 기준 (가중치)
| 지표 | 가중치 | 이유 |
|---|---|---|
| 장서 규모 | **40%** | 서가가 클수록 오배열 발생↑, 서비스 필요성↑ |
| 저자기호 결측률 낮음 | **35%** | 매칭 엔진 작동 여부에 직결 |
| 한국문학(810-819) 비중 | **25%** | 현재 MVP 범위 커버리지 |

### 데이터 출처
정보나루(data4library.kr) — 국립중앙도서관 공공 API

## 0. 환경 설정

In [ ]:
# 필요 패키지 설치 (Colab 환경)
# !pip install requests python-dotenv pandas -q

import requests
import pandas as pd
import time
import os
from collections import defaultdict

# API 키 설정 — Colab Secrets 또는 직접 입력
# Colab: 왼쪽 사이드바 🔑 Secrets에 JUNGBO_NARU_API_KEY 추가 후 아래 주석 해제
# from google.colab import userdata
# API_KEY = userdata.get('JUNGBO_NARU_API_KEY')

API_KEY = os.environ.get("JUNGBO_NARU_API_KEY", "여기에_API_키_입력")
print("API 키 설정 완료" if API_KEY and "여기에" not in API_KEY else "⚠️  API 키를 설정해주세요")

## 1. 서울 공공도서관 목록 수집

`libSrch` API로 서울 공공도서관 전체 목록과 장서 수를 가져옵니다.

In [ ]:
def fetch_lib_list(region="11", page_size=100):
    """서울(지역코드 11) 공공도서관 전체 목록 수집"""
    url = "http://data4library.kr/api/libSrch"
    libs = []
    page = 1

    while True:
        params = {
            "authKey": API_KEY,
            "region": region,
            "pageNo": page,
            "pageSize": page_size,
            "format": "json",
        }
        resp = requests.get(url, params=params, timeout=30)
        data = resp.json().get("response", {})
        libs_page = data.get("libs", [])
        if not libs_page:
            break
        for item in libs_page:
            lib = item.get("lib", {})
            try:
                book_count = int(lib.get("BookCount", 0))
            except (ValueError, TypeError):
                book_count = 0
            libs.append({
                "libCode": lib.get("libCode"),
                "libName": lib.get("libName"),
                "BookCount": book_count,
                "address": lib.get("address", ""),
            })
        page += 1
        time.sleep(0.5)

    return pd.DataFrame(libs)


df_libs = fetch_lib_list()
print(f"서울 공공도서관 총 {len(df_libs)}개 수집")
df_libs.sort_values("BookCount", ascending=False).head(10)

## 2. 후보 도서관 필터링

- **장서 15만권 이상**: 오배열이 의미 있게 발생하는 최소 규모
- **어린이·영어·청소년·교육청 도서관 제외**: ShelfAlign 대상이 일반 종합자료실

In [ ]:
EXCLUDE_KEYWORDS = ["어린이", "영어", "청소년", "유아", "교육청", "작은", "분관"]

def is_excluded(name):
    return any(kw in name for kw in EXCLUDE_KEYWORDS)

df_candidates = df_libs[
    (df_libs["BookCount"] >= 150_000) &
    (~df_libs["libName"].apply(is_excluded))
].copy().reset_index(drop=True)

print(f"후보 도서관: {len(df_candidates)}개")
df_candidates[["libName", "BookCount"]].sort_values("BookCount", ascending=False)

## 3. 도서관별 장서 샘플 수집

`itemSrch type=ALL`로 각 도서관당 **150건** 샘플 수집.

- `type=ALL`: 등록일 무관 전체 장서 대상
- 150건 표본으로 비율 추정 (오차 ±8% 수준)
- API 호출 제한: IP 미등록 기준 500회/일

In [ ]:
def fetch_sample(lib_code, n_pages=3, page_size=50, max_retries=3):
    """도서관 장서 샘플 수집 (n_pages × page_size 건)"""
    url = "http://data4library.kr/api/itemSrch"
    docs = []

    for page in range(1, n_pages + 1):
        params = {
            "authKey": API_KEY,
            "libCode": lib_code,
            "type": "ALL",
            "pageNo": page,
            "pageSize": page_size,
            "format": "json",
        }
        for attempt in range(1, max_retries + 1):
            try:
                resp = requests.get(url, params=params, timeout=120)
                data = resp.json().get("response", {})
                if data.get("errCode"):
                    raise ValueError(f"API errCode: {data['errCode']}")
                page_docs = [d.get("doc", {}) for d in data.get("docs", [])]
                docs.extend(page_docs)
                time.sleep(2)
                break
            except Exception as e:
                if attempt == max_retries:
                    raise
                time.sleep(10)

    return docs


def get_bcode(doc):
    """저자기호(book_code) 추출"""
    cns = doc.get("callNumbers") or []
    if not cns:
        return ""
    return (cns[0].get("callNumber") or {}).get("book_code", "") or ""


def analyze_sample(docs):
    """샘플에서 지표 계산"""
    n = len(docs)
    if n == 0:
        return {"n": 0, "bcode_missing_pct": None, "korlit_pct": None}

    bcode_missing = sum(1 for d in docs if not get_bcode(d))
    korlit = sum(
        1 for d in docs
        if (d.get("class_no") or "").strip()[:2] == "81"
    )

    return {
        "n": n,
        "bcode_missing_pct": round(bcode_missing / n * 100, 1),
        "korlit_pct": round(korlit / n * 100, 1),
    }

In [ ]:
# 후보 도서관 전체 수집 (API 한도 주의: 500회/일)
results = []

for _, row in df_candidates.iterrows():
    lib_code = row["libCode"]
    lib_name = row["libName"]
    try:
        docs = fetch_sample(lib_code)
        metrics = analyze_sample(docs)
        metrics["libCode"] = lib_code
        metrics["libName"] = lib_name
        metrics["BookCount"] = row["BookCount"]
        results.append(metrics)
        print(f"✓ {lib_name:<20} n={metrics['n']} | 결측={metrics['bcode_missing_pct']}% | 한국문학={metrics['korlit_pct']}%")
    except Exception as e:
        print(f"✗ {lib_name} SKIP — {type(e).__name__}")
    time.sleep(3)

df_metrics = pd.DataFrame(results)
print(f"\n수집 완료: {len(df_metrics)}개")

## 4. 종합 점수 계산

각 지표를 0~1로 정규화(min-max) 후 가중합:

```
size_score   = (장서수 - min) / (max - min)         × 0.40
bcode_score  = 1 - (결측률 / 100)                   × 0.35
korlit_score = (한국문학비중 - min) / (max - min)    × 0.25
```

In [ ]:
df = df_metrics.dropna(subset=["bcode_missing_pct", "korlit_pct"]).copy()

# 정규화
def minmax(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) if mx > mn else series * 0

df["size_score"]   = minmax(df["BookCount"])
df["bcode_score"]  = 1 - df["bcode_missing_pct"] / 100
df["korlit_score"] = minmax(df["korlit_pct"])

df["score"] = (
    0.40 * df["size_score"] +
    0.35 * df["bcode_score"] +
    0.25 * df["korlit_score"]
).round(4)

df_ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
df_ranked.index += 1

display_cols = ["libName", "BookCount", "bcode_missing_pct", "korlit_pct", "score"]
df_ranked[display_cols].rename(columns={
    "libName": "도서관",
    "BookCount": "장서수",
    "bcode_missing_pct": "저자기호결측(%)",
    "korlit_pct": "한국문학(%)",
    "score": "종합점수",
})

## 5. 결과 분석

실제 수집한 16개 도서관 결과입니다. (4개는 API 오류로 미수집)

In [ ]:
# 실측 결과 (API 재수집 없이 확인용)
actual_results = [
    {"순위": 1,  "도서관": "서울도서관",           "장서수": 618824, "저자기호결측(%)": 8.0,  "한국문학(%)": 21.3, "종합점수": 0.764},
    {"순위": 2,  "도서관": "광진정보도서관",         "장서수": 407162, "저자기호결측(%)": 2.0,  "한국문학(%)": 37.3, "종합점수": 0.664},
    {"순위": 3,  "도서관": "성동구립도서관",         "장서수": 446029, "저자기호결측(%)": 0.0,  "한국문학(%)": 22.0, "종합점수": 0.645},
    {"순위": 4,  "도서관": "금천구립가산도서관",      "장서수": 157155, "저자기호결측(%)": 0.0,  "한국문학(%)": 74.7, "종합점수": 0.601},
    {"순위": 5,  "도서관": "중랑구립정보도서관",      "장서수": 327015, "저자기호결측(%)": 0.0,  "한국문학(%)": 34.0, "종합점수": 0.589},
    {"순위": 6,  "도서관": "은평구립도서관",         "장서수": 338126, "저자기호결측(%)": 0.0,  "한국문학(%)": 30.7, "종합점수": 0.586},
    {"순위": 7,  "도서관": "관악중앙도서관",         "장서수": 330180, "저자기호결측(%)": 0.0,  "한국문학(%)": 28.0, "종합점수": 0.568},
    {"순위": 8,  "도서관": "강북문화정보도서관",      "장서수": 211869, "저자기호결측(%)": 0.0,  "한국문학(%)": 33.0, "종합점수": 0.486},
    {"순위": 9,  "도서관": "동대문구정보화도서관",    "장서수": 257493, "저자기호결측(%)": 0.0,  "한국문학(%)": 16.0, "종합점수": 0.459},
    {"순위": 10, "도서관": "서초구립반포도서관",      "장서수": 227372, "저자기호결측(%)": 0.0,  "한국문학(%)": 21.3, "종합점수": 0.453},
    {"순위": 11, "도서관": "아리랑도서관",           "장서수": 219730, "저자기호결측(%)": 0.0,  "한국문학(%)": 22.3, "종합점수": 0.451},
    {"순위": 12, "도서관": "성동구립 용답도서관",    "장서수": 179285, "저자기호결측(%)": 0.0,  "한국문학(%)": 29.7, "종합점수": 0.445},
    {"순위": 13, "도서관": "노원중앙도서관 ★",      "장서수": 186379, "저자기호결측(%)": 0.0,  "한국문학(%)": 26.0, "종합점수": 0.436},
    {"순위": 14, "도서관": "성동구립 청계도서관",    "장서수": 155872, "저자기호결측(%)": 0.0,  "한국문학(%)": 28.7, "종합점수": 0.420},
    {"순위": 15, "도서관": "도봉문화정보도서관",      "장서수": 251032, "저자기호결측(%)": 42.3, "한국문학(%)": 22.7, "종합점수": 0.331},
    {"순위": 16, "도서관": "성북정보도서관",         "장서수": 280003, "저자기호결측(%)": 54.0, "한국문학(%)": 10.7, "종합점수": 0.268},
]

df_actual = pd.DataFrame(actual_results).set_index("순위")
df_actual.style.background_gradient(subset=["종합점수"], cmap="RdYlGn")

## 6. 주요 해석

### 추천 상위 3개관

| 도서관 | 근거 |
|---|---|
| **서울도서관 (1위)** | 장서 61만으로 압도적 규모. 서울 대표 도서관으로 공모전 임팩트 최대 |
| **광진정보도서관 (2위)** | 결측 2%·한국문학 37%로 균형 최상. 40만 장서 |
| **성동구립도서관 (3위)** | 결측 0%·44만 장서. 데이터 품질 완벽 |

### 제외 권고
- **성북정보도서관** (저자기호 결측 54%) → 매칭 엔진 정상 작동 불가
- **도봉문화정보도서관** (저자기호 결측 42%) → 동일 이유

### 현재 MVP 노원중앙도서관 (13위)
데이터 품질(결측 0%)은 우수하나 장서 규모에서 상위관 대비 열세.

### 분석 한계
- **표본 편향**: 최신 등록순 정렬 150건 샘플 → 저자기호 결측률 실제보다 낮게 추정될 수 있음
- **이용자 수 미반영**: 실제 방문객·대출 수 데이터 없어 장서 규모로 대체
- **4개 미수집**: 마포중앙·서대문구립이진아·금천구립독산·성동구립금호 API 오류